In [1]:
import sys
from pathlib import Path
_r = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'config.py').exists())
sys.path.insert(0, str(_r / 'scripts'))

from notebook_init import setup
cfg, PATHS, POPULATIONS, HARD_FILTERS, SITUATIONAL_FILTERS, ML = setup()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
print("All imports successful!")
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from stats import compute_snp_stats_vectorized
from ml_training import train_random_forest, train_xgboost, train_logistic_regression, evaluate_model, cross_validate_model, get_feature_importances, save_model
from statsmodels.stats.multitest import fdrcorrection
import os, time
import xgboost as xgb


Project root : /home/ibmelab/Projects/AISNP_Research
genomes_data : /mnt/data/aisnp_data/1000genomes/
output root  : output
All imports successful!


In [2]:
output_dir = PATHS.outputs_dir("statistical_v2/05c_fst_stat_training")
import os
os.makedirs(str(output_dir), exist_ok=True)
print(f'Output dir: {output_dir}')

Output dir: /mnt/data/aisnp_data/1000genomes/outputs/statistical_v2/05c_fst_stat_training


## Step 1: Load FST-Selected Genotype Matrix

Input: `PATHS.ML_DATA` produced by `04b_fst_and_pca` + `05b` extraction.

In [3]:
ML_DATA_PATH = str(PATHS.ML_DATA)
print(f'Loading FST ML data: {ML_DATA_PATH}')
fst_df = pd.read_csv(ML_DATA_PATH)

snp_columns = [c for c in fst_df.columns if c not in ['sample', 'pop']]
populations = fst_df['pop']
print(f'Shape: {fst_df.shape}')
print(f'FST SNPs: {len(snp_columns)}')
print(f'Populations: {populations.value_counts().to_dict()}')


Loading FST ML data: /mnt/data/aisnp_data/1000genomes/outputs/fst_only/04b_fst_and_pca/ml_data_with_pop.csv
Shape: (504, 2510)
FST SNPs: 2508
Populations: {'CN': 208, 'SEA': 192, 'JPT': 104}


## Step 2: Statistical Tests on FST-Selected SNPs

Runs chi², MI, and KL divergence on the FST subset only (~2500 SNPs).
Much faster than running on the full 600k set.

In [4]:
STATS_CACHE = str(output_dir / 'consensus_snp_stats.csv')
os.makedirs(str(output_dir), exist_ok=True)

if os.path.exists(STATS_CACHE):
    print(f'Loading cached stats: {STATS_CACHE}')
    stats_df = pd.read_csv(STATS_CACHE)
else:
    print(f'Computing stats for {len(snp_columns)} FST SNPs...')
    t0 = time.time()
    stats_df = compute_snp_stats_vectorized(fst_df, populations)
    print(f'Done in {time.time()-t0:.1f}s')
    stats_df.to_csv(STATS_CACHE, index=False)
    print(f'Saved: {STATS_CACHE}')

print(f'\nStats shape: {stats_df.shape}')
print(stats_df.describe().round(4))


Computing stats for 2508 FST SNPs...
Vectorised stats: 2,508 SNPs × 504 samples × 3 populations
Done.
Done in 0.0s
Saved: /mnt/data/aisnp_data/1000genomes/outputs/statistical_v2/05c_fst_stat_training/consensus_snp_stats.csv

Stats shape: (2508, 5)
            chi2  chi2_pvalue  mutual_information  kl_divergence
count  2508.0000    2508.0000           2508.0000      2508.0000
mean     42.9725       0.0000              0.0424         0.2933
std      17.6892       0.0003              0.0159         0.3280
min      12.9790       0.0000              0.0135         0.0518
25%      30.7561       0.0000              0.0321         0.1271
50%      38.7617       0.0000              0.0392         0.1716
75%      50.9234       0.0000              0.0490         0.2816
max     175.0819       0.0114              0.1922         2.8563


## Step 3: Multiple Testing Correction

In [5]:
alpha = 0.05
n_tests = len(stats_df)
bonf_alpha = alpha / n_tests

chi2_pvals = stats_df['chi2_pvalue'].fillna(1).values
chi2_reject, chi2_qvals = fdrcorrection(chi2_pvals, alpha=alpha)
stats_df['chi2_qvalue'] = chi2_qvals
stats_df['chi2_sig_fdr'] = chi2_reject
stats_df['chi2_sig_bonf'] = stats_df['chi2_pvalue'] < bonf_alpha

print(f'FST SNPs: {n_tests}')
print(f'chi² FDR significant: {chi2_reject.sum()}')
print(f'chi² Bonferroni significant: {stats_df["chi2_sig_bonf"].sum()}')


FST SNPs: 2508
chi² FDR significant: 2508
chi² Bonferroni significant: 2150


## Step 4: Select Consensus SNPs

SNPs must pass chi² FDR **and** rank in the top 50% for MI and KL divergence.

In [6]:
top_n = max(50, len(stats_df) // 2)

sig_chi2 = set(stats_df[stats_df['chi2_sig_fdr']]['snp_id'])
sig_mi   = set(stats_df.nlargest(top_n, 'mutual_information')['snp_id'])
sig_kl   = set(stats_df.nlargest(top_n, 'kl_divergence')['snp_id'])

from collections import Counter
all_sig = list(sig_chi2) + list(sig_mi) + list(sig_kl)
snp_counts = Counter(all_sig)
snps_all3 = [s for s, c in snp_counts.items() if c == 3]
snps_ge2  = [s for s, c in snp_counts.items() if c >= 2]

print(f'chi² FDR significant : {len(sig_chi2)}')
print(f'Top-{top_n} MI        : {len(sig_mi)}')
print(f'Top-{top_n} KL        : {len(sig_kl)}')
print(f'Pass all 3 tests      : {len(snps_all3)}')
print(f'Pass ≥2 tests         : {len(snps_ge2)}')

if len(snps_all3) >= 25:
    consensus_snps = snps_all3
    consensus_label = 'all 3 tests'
elif len(snps_ge2) >= 25:
    consensus_snps = snps_ge2
    consensus_label = '≥2 tests'
else:
    consensus_snps = list(sig_chi2)
    consensus_label = 'chi² FDR only'

# Rank by composite score
stats_df['rank_chi2'] = stats_df['chi2'].rank(ascending=False)
stats_df['rank_mi']   = stats_df['mutual_information'].rank(ascending=False)
stats_df['rank_kl']   = stats_df['kl_divergence'].rank(ascending=False)
stats_df['composite_rank'] = (stats_df['rank_chi2'] + stats_df['rank_mi'] + stats_df['rank_kl']) / 3

consensus_df = stats_df[stats_df['snp_id'].isin(consensus_snps)].sort_values('composite_rank')
available_snps = [s for s in consensus_df['snp_id'] if s in fst_df.columns]

print(f'\nUsing: {consensus_label}  ({len(available_snps)} SNPs available in matrix)')
display(consensus_df[['snp_id','chi2','chi2_pvalue','mutual_information','kl_divergence','composite_rank']].head(15))


chi² FDR significant : 2508
Top-1254 MI        : 1254
Top-1254 KL        : 1254
Pass all 3 tests      : 1003
Pass ≥2 tests         : 1505

Using: all 3 tests  (1003 SNPs available in matrix)


,snp_id,chi2,chi2_pvalue,mutual_information,kl_divergence,composite_rank
5,"4:17813761[b37]G,A",175.081940,8.483677e-37,0.192157,2.268300,1.666667
8,"1:12387655[b37]G,A",159.962067,1.489573e-33,0.173566,2.211642,2.666667
64,"14:96938945[b37]A,T",138.716919,5.312759e-29,0.160640,2.856276,4.333333
30,"5:41181491[b37]G,T",142.028442,1.038326e-29,0.155135,1.900760,6.000000
268,"11:112053732[b37]T,A",102.374985,3.069877e-21,0.118875,2.152359,13.333333
358,"1:102457870[b37]G,A",114.027145,1.006583e-23,0.094362,1.962984,13.666667
530,"20:33681323[b37]T,C",109.724236,8.333019e-23,0.090733,1.901932,16.000000
440,"6:170619277[b37]G,A",109.724236,8.333019e-23,0.090733,1.888224,16.666667
368,"6:82912256[b37]T,A",107.634872,2.324359e-22,0.121366,1.422121,23.666667
624,"1:241502631[b37]T,C",101.172318,5.536606e-21,0.083539,1.739370,29.000000


## Step 5: Prepare Training Data

In [7]:
X_consensus = fst_df[available_snps].copy()
# Sanitize column names for XGBoost
X_consensus.columns = [
    c.replace('[','_').replace(']','_').replace('<','_') for c in X_consensus.columns
]
y = populations

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X_consensus, y, test_size=ML.TEST_SIZE, stratify=y, random_state=ML.RANDOM_STATE
)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

print(f'Consensus SNPs: {X_consensus.shape[1]}')
print(f'Train: {len(X_train)}  Test: {len(X_test)}')


Consensus SNPs: 1003
Train: 403  Test: 101


## Step 6: Train Models on FST+Statistical Consensus SNPs

In [8]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Random Forest
rf = RandomForestClassifier(n_estimators=ML.RF_N_ESTIMATORS, random_state=ML.RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
rf_acc = rf.score(X_test, y_test)
print(f'Random Forest:        {rf_acc:.4f}')

# XGBoost
xgb_clf = xgb.XGBClassifier(
    n_estimators=ML.XGB_N_ESTIMATORS, max_depth=ML.XGB_MAX_DEPTH,
    learning_rate=ML.XGB_LEARNING_RATE, random_state=ML.RANDOM_STATE,
    n_jobs=-1, verbosity=0,
)
xgb_clf.fit(X_train, y_train_enc)
xgb_acc = xgb_clf.score(X_test, y_test_enc)
print(f'XGBoost:              {xgb_acc:.4f}')

# Logistic Regression
lr = LogisticRegression(max_iter=ML.LR_MAX_ITER, solver=ML.LR_SOLVER, random_state=ML.RANDOM_STATE)
lr.fit(X_train_sc, y_train)
lr_acc = lr.score(X_test_sc, y_test)
print(f'Logistic Regression:  {lr_acc:.4f}')


Random Forest:        0.9604
XGBoost:              0.9307
Logistic Regression:  0.9802


## Step 7: Performance vs Feature Count (25–50 SNPs)

In [9]:
cv = StratifiedKFold(n_splits=ML.CV_FOLDS, shuffle=True, random_state=ML.RANDOM_STATE)
snp_counts_range = [n for n in [5,10,15,20,25,30,35,40,45,50] if n <= len(available_snps)]

# --- RF importance ---
rf_imp = pd.DataFrame({
    'feature': X_consensus.columns,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

rf_sweep = []
print('RF sweep:')
for n in snp_counts_range:
    top = rf_imp.head(n)['feature'].tolist()
    clf = RandomForestClassifier(n_estimators=100, random_state=ML.RANDOM_STATE, n_jobs=-1)
    scores = cross_val_score(clf, X_consensus[top], y, cv=cv, scoring='accuracy')
    rf_sweep.append({'n_snps': n, 'mean_acc': scores.mean(), 'std_acc': scores.std()})
    print(f'  {n:3d} SNPs: {scores.mean():.4f} ± {scores.std():.4f}')
rf_sweep_df = pd.DataFrame(rf_sweep)

# --- LR importance ---
lr_imp = pd.DataFrame({
    'feature': X_consensus.columns,
    'importance': np.abs(lr.coef_).mean(axis=0),
}).sort_values('importance', ascending=False).reset_index(drop=True)

lr_sweep = []
print('\nLR sweep:')
for n in snp_counts_range:
    top = lr_imp.head(n)['feature'].tolist()
    X_sub_sc = scaler.fit_transform(X_consensus[top])
    clf = LogisticRegression(max_iter=ML.LR_MAX_ITER, solver=ML.LR_SOLVER, random_state=ML.RANDOM_STATE)
    scores = cross_val_score(clf, X_sub_sc, y, cv=cv, scoring='accuracy')
    lr_sweep.append({'n_snps': n, 'mean_acc': scores.mean(), 'std_acc': scores.std()})
    print(f'  {n:3d} SNPs: {scores.mean():.4f} ± {scores.std():.4f}')
lr_sweep_df = pd.DataFrame(lr_sweep)

# Combined comparison table
sweep_combined = rf_sweep_df.rename(columns={'mean_acc': 'rf_acc', 'std_acc': 'rf_std'}).merge(
    lr_sweep_df.rename(columns={'mean_acc': 'lr_acc', 'std_acc': 'lr_std'}), on='n_snps'
)
display(sweep_combined.round(4))


RF sweep:
    5 SNPs: 0.7063 ± 0.0443
   10 SNPs: 0.7540 ± 0.0206
   15 SNPs: 0.8115 ± 0.0487
   20 SNPs: 0.7877 ± 0.0389
   25 SNPs: 0.8055 ± 0.0240
   30 SNPs: 0.8413 ± 0.0176
   35 SNPs: 0.8810 ± 0.0215
   40 SNPs: 0.8968 ± 0.0248
   45 SNPs: 0.9028 ± 0.0097
   50 SNPs: 0.9206 ± 0.0166

LR sweep:
    5 SNPs: 0.6566 ± 0.0433
   10 SNPs: 0.7202 ± 0.0278
   15 SNPs: 0.8055 ± 0.0244
   20 SNPs: 0.8373 ± 0.0202
   25 SNPs: 0.8294 ± 0.0210
   30 SNPs: 0.8511 ± 0.0181
   35 SNPs: 0.8690 ± 0.0326
   40 SNPs: 0.8928 ± 0.0258
   45 SNPs: 0.8987 ± 0.0267
   50 SNPs: 0.9007 ± 0.0202


,n_snps,rf_acc,rf_std,lr_acc,lr_std
0,5,0.7063,0.0443,0.6566,0.0433
1,10,0.7540,0.0206,0.7202,0.0278
2,15,0.8115,0.0487,0.8055,0.0244
3,20,0.7877,0.0389,0.8373,0.0202
4,25,0.8055,0.0240,0.8294,0.0210
5,30,0.8413,0.0176,0.8511,0.0181
6,35,0.8810,0.0215,0.8690,0.0326
7,40,0.8968,0.0248,0.8928,0.0258
8,45,0.9028,0.0097,0.8987,0.0267
9,50,0.9206,0.0166,0.9007,0.0202


## Step 8: Save Results

In [10]:
# Model comparison
results_df = pd.DataFrame([
    {'model': 'Random Forest',       'accuracy': rf_acc},
    {'model': 'XGBoost',             'accuracy': xgb_acc},
    {'model': 'Logistic Regression', 'accuracy': lr_acc},
])
results_df.to_csv(str(output_dir / 'statistical_ml_results.csv'), index=False)
print('Model comparison saved.')
display(results_df)

# RF feature importances + top-50 matrix
rf_imp.to_csv(str(output_dir / 'rf_feature_importances.csv'), index=False)
rf_top50 = rf_imp.head(50)['feature'].tolist()
rf_top50_df = fst_df[['sample','pop'] + [s for s in rf_top50 if s in fst_df.columns]]
rf_top50_df.to_csv(str(output_dir / 'fst_stat_rf_top50_ml_data.csv'), index=False)
print(f'RF top-50 matrix: {rf_top50_df.shape}')

# LR feature importances + top-50 matrix
lr_imp.to_csv(str(output_dir / 'lr_feature_importances.csv'), index=False)
lr_top50 = lr_imp.head(50)['feature'].tolist()
lr_top50_df = fst_df[['sample','pop'] + [s for s in lr_top50 if s in fst_df.columns]]
lr_top50_df.to_csv(str(output_dir / 'fst_stat_lr_top50_ml_data.csv'), index=False)
print(f'LR top-50 matrix: {lr_top50_df.shape}')

# Sweep results
rf_sweep_df.to_csv(str(output_dir / 'rf_feature_size_results.csv'), index=False)
lr_sweep_df.to_csv(str(output_dir / 'lr_feature_size_results.csv'), index=False)
print(f'Sweep results saved to: {output_dir}')


Model comparison saved.


,model,accuracy
0,Random Forest,0.960396
1,XGBoost,0.930693
2,Logistic Regression,0.980198


RF top-50 matrix: (504, 2)
LR top-50 matrix: (504, 2)
Sweep results saved to: /mnt/data/aisnp_data/1000genomes/outputs/statistical_v2/05c_fst_stat_training


## Summary

In [11]:
print('='*70)
print('05c FST + STATISTICAL TRAINING SUMMARY')
print('='*70)
print(f'  FST SNPs input       : {len(snp_columns)}')
print(f'  After stat selection : {len(available_snps)}  ({consensus_label})')
print(f'  chi² FDR sig         : {len(sig_chi2)}')
print(f'  RF accuracy          : {rf_acc:.4f}')
print(f'  XGBoost accuracy     : {xgb_acc:.4f}')
print(f'  LR accuracy          : {lr_acc:.4f}')
print('='*70)


05c FST + STATISTICAL TRAINING SUMMARY
  FST SNPs input       : 2508
  After stat selection : 1003  (all 3 tests)
  chi² FDR sig         : 2508
  RF accuracy          : 0.9604
  XGBoost accuracy     : 0.9307
  LR accuracy          : 0.9802
